# 第 2 单元第 3 节：高级 Evaluation 技术

在第 1 节和第 2 节中，我们使用了**最终答案 Evaluation** — 仅根据真实答案对 Agent 的最后一条消息进行评分。这是一个很好的起点，但它是一个黑匣子：它告诉您 Agent 是否获得了正确的答案，而不是它如何到达那里。

随着Agent变得越来越复杂，通过多步骤推理、子 Agent 路由和 HITL 验证步骤，仅最终答案评估就留下了太多未检测到的故障模式。响应可能因错误的原因而正确，或者采取极其低效的路径来达到目标​​。

在本节中，我们将添加三种互补的 Evaluation 技术，让您了解 Agent 的内部结构：

|评估类型 |它测量什么 |关键问题|
|---|---|---|
| **最终答案** _（第 1 节）_ |上次回复的正确性 | Agent 给出了正确答案吗？ |
| **单步** |一个孤立的决策点 | Agent 在步骤 X 中做出了正确的选择吗？ |
| **Trajectory** |完整的Tool Calling序列+执行路径 | Agent 是否采取了正确的路径来找到答案？ |
| **多轮模拟** |完整的多消息对话 | Agent 在现实交互中的表现如何？ |

每种技术都针对 Agent 行为的不同级别。它们共同构成了一个全面的评估套件。

## 设置

In [ ]:
import uuid
from dotenv import load_dotenv

load_dotenv()

---
## 1.单步Evaluation

单步 Evaluation 在 Agent 中隔离一个 **单个决策点** 并独立评估它 - 就像图中一个节点的单元测试一样。

<div align="center">
    <img src="../../images/single-step-eval.png"  width="800">
</div>



**我们正在测试的内容：** Supervisor 的路由决策。给定客户问题（身份已验证），Supervisor 是否正确路由到 `database_specialist` 与 `documentation_specialist`？

错误的路由会导致调用错误的工具，从而导致错误的答案 - 但如果 Agent 恢复，仅最终答案评估可能不会显示这一点。单步评估直接捕获根本原因。

**为什么要隔离？** 通过将已知的 `customer_id` 直接注入状态，我们完全绕过了 HITL 验证步骤。这让我们可以单独测试路由逻辑——没有来自上游分类器的噪音。

### 1.1 初始化Supervisor Agent

In [ ]:
from agents.docs_agent import create_docs_agent
from agents.sql_agent import create_sql_agent
from agents.supervisor_agent import create_supervisor_agent
from agents.supervisor_hitl_agent import IntermediateState

sql_agent = create_sql_agent()
docs_agent = create_docs_agent()

# IntermediateState extends MessagesState with a `customer_id` field.
# We must pass it here so the supervisor's state schema includes `customer_id` —
# otherwise the value we inject at invoke time gets silently dropped and the
# dynamic prompt never receives it.
supervisor = create_supervisor_agent(
    db_agent=sql_agent,
    docs_agent=docs_agent,
    state_schema=IntermediateState,
)

### 1.2 创建Dataset

我们的 dataset 包含与预期路由目标（`database_specialist` 或 `documentation_specialist`）配对的问题。每个示例还包括一个 `customer_id` — 传递到状态以模拟已验证的客户，仅保持路由逻辑的真正单元测试。

> **注意：** Supervisor 工具名称来自 `supervisor_agent.py`：`"database_specialist"` 包装 SQL/DB Agent，`"documentation_specialist"` 包装文档 Agent。

In [ ]:
from langsmith import Client

client = Client()

examples = [
    # Database questions — require database_specialist
    {
        "question": "What's the status of my most recent order?",
        "customer_id": "CUST-001",
        "route": "database_specialist",
    },
    {
        "question": "How much have I spent on monitors this year?",
        "customer_id": "CUST-007",
        "route": "database_specialist",
    },
    {
        "question": "How many orders did I place in the last 6 months?",
        "customer_id": "CUST-012",
        "route": "database_specialist",
    },
    # Documentation questions — require documentation_specialist
    {
        "question": "What's the return policy for opened electronics?",
        "customer_id": "CUST-001",
        "route": "documentation_specialist",
    },
    {
        "question": "Is the Sony WH-1000XM5 still under warranty?",
        "customer_id": "CUST-007",
        "route": "documentation_specialist",
    },
    {
        "question": "What accessories are compatible with TechHub laptops?",
        "customer_id": "CUST-012",
        "route": "documentation_specialist",
    },
]

dataset_name = f"techhub-single-step-routing-{uuid.uuid4()}"

dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Single-step eval: supervisor routing decisions (database_specialist vs documentation_specialist)",
)

client.create_examples(
    dataset_id=dataset.id,
    inputs=[
        {"question": ex["question"], "customer_id": ex["customer_id"]}
        for ex in examples
    ],
    outputs=[{"route": ex["route"]} for ex in examples],
)

print(f"Dataset: {dataset.url}")

### 1.3 目标函数

目标函数使用 dataset 中预先验证的 `customer_id` 调用 Supervisor，然后从响应中提取**第一个Tool Calling名称** - 这就是路由决策。

我们传递 `interrupt_before=["tools"]`，因此在 LLM 选择工具后、子 Agent 实际运行之前，执行立即停止。这使得评估快速且真正隔离——我们正在测试决策，而不是下游结果。

In [ ]:
def run_supervisor_routing(inputs: dict) -> dict:
    """Invoke supervisor and capture the routing decision before any tool executes."""
    result = supervisor.invoke(
        {
            "messages": [{"role": "user", "content": inputs["question"]}],
            "customer_id": inputs["customer_id"],  # per-example, not hardcoded
        },
        config={"configurable": {"thread_id": str(uuid.uuid4())}},
        interrupt_before=["tools"],  # stop after the LLM decides, before sub-agent runs
    )
    # The last AIMessage with tool_calls contains the routing decision
    for msg in result["messages"]:
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            return {"route": msg.tool_calls[0]["name"]}
    return {"route": None}

In [ ]:
# Quick test on one example
test_output = run_supervisor_routing(
    {"question": "What's the status of my recent order?", "customer_id": "CUST-001"}
)
print(f"Routing decision: {test_output['route']}")

### 1.4 Evaluator

一个简单的相等性检查 - 路由决策应该与预期的子 Agent 完全匹配。

In [ ]:
def correct_route(outputs: dict, reference_outputs: dict) -> bool:
    """Check if the supervisor routed to the correct sub-agent."""
    return outputs["route"] == reference_outputs["route"]

### 1.5 运行Experiment

In [ ]:
results = client.evaluate(
    run_supervisor_routing,
    data=dataset_name,
    evaluators=[correct_route],
    experiment_prefix="supervisor-routing-single-step",
    description="Unit test: does the supervisor route to the correct sub-agent for each question type?",
    max_concurrency=3,
)

### 1.6 分析结果

打开上面的 experiment 链接并检查任何错误路由的示例。 `correct_route = False` 结果准确地告诉您哪个问题导致了错误的路由决策——这是一个比最终答案失败更有针对性的信号。

**要查找的常见故障模式：**
- 模棱两可的问题（例如，“告诉我有关我的笔记本电脑的信息”），可能会转到子 Agent
- 在一条消息中同时提及帐户信息和产品信息的问题
- 措辞不当、意图不明确的问题

如果发现故障，修复的目标是：更新 Supervisor 的系统提示符而不是整个管道。

---
## 2.Trajectory Evaluation

Trajectory Evaluation 检查 Agent 运行的**完整执行路径** — 访问了哪些节点，调用了哪些工具，以及 Agent 是否遵循预期路径得到答案。

<div align="center">
    <img src="../../images/trajectory-eval.png"  width="800">
</div>

正确的最终答案并不能保证有效或正确的路径。例如，Agent 可能：
- 当一个 JOIN 查询就足够时，调用 `execute_sql` 3 次
- 跳过特定于帐户的查询的验证步骤
- 路由到错误的子Agent，得到不好的结果，然后通过尝试正确的子来恢复

**我们正在测试的内容：** 完整的 `supervisor_hitl_agent`（带有 SQL Agent） — 包括验证、路由和Tool Calling序列。

**HITL处理：**部分问题需要身份验证。目标函数检测 Agent 是否在 interrupt 处暂停（等待电子邮件），并使用来自 dataset 的每个示例电子邮件继续 - 问题本身没有包含电子邮件。

### 2.1 初始化完整的HITL Agent

In [ ]:
from agents.supervisor_hitl_agent import create_supervisor_hitl_agent

# Use the SQL agent (from Section 2) with the full HITL pipeline
# use_checkpointer=True is required to call get_state() for HITL handling
agent = create_supervisor_hitl_agent(
    db_agent=create_sql_agent(),
    docs_agent=create_docs_agent(),
)

### 2.2 创建Dataset

每个示例指定：
- `question` — 客户的消息
- `customer_email` — 用于在触发时恢复 HITL interrupt（不在问题中）
- `expects_verification` — 我们是否期望运行 HITL 验证步骤
- `expected_tools` - 我们期望在各个级别调用的所有工具：Supervisor 工具（`database_specialist`、`documentation_specialist`）**和**子 Agent 工具（`execute_sql`、`search_policy_documents`、 `search_product_documents`)

> **为什么要包含子 Agent 工具？** 子Agent在工具包装器内作为 Python 函数进行调用，因此它们的内部Tool Calling（例如 `execute_sql`）不会出现在父图的消息状态中。为了查看它们，我们使用**基于运行的评估器**，通过子运行遍历完整的 LangSmith 运行树 - 与 `count_total_tool_calls_evaluator` 目录中的 `evaluators/` 使用的模式相同。

最后一个示例旨在对 **效率** evaluator 进行压力测试：一个多部分数据库问题，可能会导致 SQL Agent 进行冗余 `execute_sql` 调用（以多种方式查询相同的数据），而不是合并为单个高效查询。

In [ ]:
from pprint import pprint

examples = [
    # Requires verification + DB query only
    {
        "question": "What were the items in my most recent order and what did each cost?",
        "customer_email": "sarah.chen@gmail.com",
        "expects_verification": True,
        "expected_tools": ["database_specialist", "execute_sql"],
    },
    # No verification needed + docs query only (return policy)
    {
        "question": "What's the return policy for opened electronics?",
        "customer_email": None,
        "expects_verification": False,
        "expected_tools": ["documentation_specialist", "search_policy_docs"],
    },
    # Requires verification + BOTH sub-agents called (cross-domain query)
    {
        "question": "Are any of my recent orders for items that are still within the return policy window?",
        "customer_email": "emily.rodriguez@icloud.com",
        "expects_verification": True,
        "expected_tools": [
            "database_specialist",
            "execute_sql",
            "documentation_specialist",
            "search_policy_docs",
        ],
    },
    # Requires verification + DB aggregation
    {
        "question": "How much have I spent on keyboards in total?",
        "customer_email": "james.nguyen@yahoo.com",
        "expects_verification": True,
        "expected_tools": ["database_specialist", "execute_sql"],
    },
    # No verification needed + product docs
    {
        "question": "What are the specs of the Sony WH-1000XM5?",
        "customer_email": None,
        "expects_verification": False,
        "expected_tools": ["documentation_specialist", "search_product_docs"],
    },
    # Efficiency stress test: complex multi-part DB question.
    # An efficient SQL agent should answer this in 1-2 queries.
    # A naive agent might call execute_sql separately for each part —
    # designed to trigger no_repeated_calls to fail and surface the redundancy.
    {
        "question": (
            "Give me a summary of my order history: how many orders have I placed, "
            "what's my total spend, and which product category do I buy most often?"
        ),
        "customer_email": "sarah.chen@gmail.com",
        "expects_verification": True,
        "expected_tools": ["database_specialist", "execute_sql"],
    },
]

pprint(examples[-1])

In [ ]:
trajectory_dataset_name = f"techhub-trajectory-eval-{uuid.uuid4()}"

trajectory_dataset = client.create_dataset(
    dataset_name=trajectory_dataset_name,
    description="Trajectory eval: expected tool call sequences for full HITL agent",
)

client.create_examples(
    dataset_id=trajectory_dataset.id,
    inputs=[
        {"question": ex["question"], "customer_email": ex["customer_email"]}
        for ex in examples
    ],
    outputs=[
        {
            "expects_verification": ex["expects_verification"],
            "expected_tools": ex["expected_tools"],
        }
        for ex in examples
    ],
)

print(f"Dataset: {trajectory_dataset.url}")

### 2.3 目标函数

目标函数调用完整的 Agent 并干净地处理 HITL interrupt：

1. 根据客户的问题运行Agent
2、检查Agent是否暂停在interrupt（需要验证身份）
3. 如果是这样，请使用 dataset 中的 `customer_email` 继续 — **问题中没有电子邮件**
4. 返回Evaluation的完整消息历史记录

In [ ]:
from langgraph.types import Command


def run_agent_trajectory(inputs: dict) -> dict:
    """Run the full HITL agent and return the complete message history."""
    thread_id = str(uuid.uuid4())
    config = {"configurable": {"thread_id": thread_id}}

    # Initial invocation
    agent.invoke(
        {"messages": [{"role": "user", "content": inputs["question"]}]},
        config=config,
    )

    # If the agent paused at the email verification interrupt, resume with the
    # customer email from the dataset (not embedded in the question).
    # Guard: only resume if customer_email is provided — calling Command(resume=None)
    # triggers a LangGraph bug. If state.next is True but no email is available,
    # the verification check will correctly fail (no "Verified!" message in state).
    state = agent.get_state(config)
    if state.next and inputs.get("customer_email"):
        agent.invoke(Command(resume=inputs["customer_email"]), config=config)

    # Return the full final message history
    final_state = agent.get_state(config)
    return {"messages": final_state.values["messages"]}

In [ ]:
# Quick test
test_result = run_agent_trajectory(
    {
        "question": "What's the return policy for opened electronics?",
        "customer_email": None,
    }
)

for msg in test_result["messages"]:
    msg.pretty_print()

### 2.4 评估者

我们将使用四个评估器，共同给出 trajectory 质量的完整画面：

1. **验证步骤正确** — HITL 验证节点是否在应有的时间运行？
2. **名为**的预期工具 — 是否使用了所有预期工具，包括 Agent 子工具？
3. **无重复调用** — Agent 是否避免了重复的Tool Calling？ （效率）
4. **[可选] LLM-as-judge trajectory** — 使用 `agentevals` 进行灵活的质量评估

评估器 2 和 3 使用 **`Run` 对象** — LangSmith trace — 而不是消息列表。这允许访问完整的运行树，包括来自子Agent的子运行，因此我们可以检查从未出现在父图的消息状态中的 `execute_sql` 调用。

第一个 evaluator 是确定性的且便宜。 LLM 法官能够捕捉到其他人无法捕捉到的细微问题。

In [ ]:
# Evaluator #1: Did the agent correctly route the question for identity verification?
def verification_step_correct(outputs: dict, reference_outputs: dict) -> dict:
    """Check whether the verify_customer node ran (or was correctly skipped).

    The verify_customer node adds an AIMessage containing 'Verified!' when it
    successfully validates a customer. We use this as a proxy to detect whether
    the HITL verification path was triggered.
    """
    messages = outputs["messages"]
    verified = any(
        "Verified!" in (getattr(msg, "content", "") or "") for msg in messages
    )
    return {
        "key": "verification_correct",
        "score": verified == reference_outputs["expects_verification"],
    }

In [ ]:
from collections import Counter

from langsmith.schemas import Run


# Helper
def extract_all_tool_names(run: Run) -> list[str]:
    """Recursively collect every tool call name from a run and all its child runs."""
    names = []
    if run.run_type == "tool":
        names.append(run.name)
    for child in run.child_runs or []:
        names.extend(extract_all_tool_names(child))
    return names


# Evaluator #2: Did the agent call all expected tools?
def expected_tools_called(run: Run, reference_outputs: dict) -> dict:
    """Check that all expected tools appear somewhere in the run tree.

    Because sub-agent tool calls (e.g. execute_sql) only appear as child runs in the
    LangSmith trace — not in the parent graph's message state — we traverse the full
    run tree via the Run object rather than inspecting messages.
    """
    actual = extract_all_tool_names(run)
    expected = reference_outputs["expected_tools"]
    missing = [t for t in expected if t not in actual]
    return {
        "key": "expected_tools_called",
        "score": len(missing) == 0,
        "comment": f"Missing: {missing}" if missing else "All expected tools called",
    }


# Evaluator #3: Did the agent avoid making redundant tool calls?
def no_repeated_calls(run: Run) -> dict:
    """Check that no tool was called more than once (efficiency guard).

    Repeated calls to the same tool — especially execute_sql — signal that the agent
    is making redundant queries instead of consolidating into a single efficient one.
    This connects directly to the SQL agent improvement from Section 2: the rigid
    db_agent often made multiple tool calls for questions a single SQL query could answer.
    """
    actual = extract_all_tool_names(run)
    counts = Counter(actual)
    repeated = {name: count for name, count in counts.items() if count > 1}
    return {
        "key": "no_repeated_calls",
        "score": len(repeated) == 0,
        "comment": str(repeated) if repeated else "No repeated calls",
    }

#### [可选] LLM 作为法官 Trajectory Evaluator

`agentevals` 软件包提供基于 LLM 的 trajectory evaluator，无需严格的参考即可评估总体路径质量。它对于捕获确定性评估器无法捕获的问题特别有用，例如使用格式不正确的查询调用正确的工具。

> **权衡：**确定性评估器快速、廉价且一致，但需要明确的基本事实。 LLM 判断很灵活，可以捕捉细微的问题，但速度较慢且确定性较差。在实践中，两者都使用。

In [ ]:
from agentevals.trajectory.llm import (
    TRAJECTORY_ACCURACY_PROMPT,
    create_trajectory_llm_as_judge,
)
from config import DEFAULT_MODEL

# Create the underlying agentevals judge
_trajectory_judge = create_trajectory_llm_as_judge(
    model=DEFAULT_MODEL,
    prompt=TRAJECTORY_ACCURACY_PROMPT,
)


def trajectory_accuracy(outputs: dict, **kwargs) -> dict:
    """Wrap agentevals LLM judge to extract messages from our outputs dict."""
    return _trajectory_judge(outputs=outputs["messages"])

### 2.5 运行Experiment

In [ ]:
trajectory_results = client.evaluate(
    run_agent_trajectory,
    data=trajectory_dataset_name,
    evaluators=[
        verification_step_correct,
        expected_tools_called,
        no_repeated_calls,
        trajectory_accuracy,
    ],
    experiment_prefix="full-agent-trajectory",
    description="Trajectory eval: verification step, tool call correctness, efficiency, and path quality",
    max_concurrency=2,
)

### 2.6 分析结果

在 LangSmith 中，一起查看四个指标：

- **`verification_correct`** — 如果为 False，则 HITL 步骤在不应跳过（或不必要地触发）时被跳过。这是与安全相关的故障。
- **`expected_tools_called`** — 如果为 False，则 Agent 无法收集正确的信息。检查 `comment` 字段以查看缺少哪些工具。由于此 evaluator 遍历完整运行树，因此它捕获丢失的子 Agent 调用（例如，`execute_sql` 从未运行）。
- **`no_repeated_calls`** — 如果为 False，Agent 会进行冗余Tool Calling。检查 `comment` 字段哪些工具被重复以及重复了多少次。这直接连接到第 2 部分的故事：旧的 `db_agent` 将为每条信息调用单独的刚性工具，而 SQL Agent 应该合并为更少的查询。
- **`trajectory_accuracy`** — LLM 法官对路径质量的整体判断。

**要寻找的东西：**
- **示例 3**（跨域查询）：Supervisor 是否同时调用了 `database_specialist` 和 `documentation_specialist`？如果只调用一个，`expected_tools_called` 将失败。
- **示例 6**（效率压力测试）：旨在揭示 SQL Agent 是否对多部分问题进行冗余的 `execute_sql` 调用。如果 `no_repeated_calls` 在此失败，`comment` 将准确显示哪个工具被冗余调用以及调用了多少次 - 为您提供有关如何改进 Agent 的 SQL 生成或提示的具体指导。

---
## 3. 多回合模拟

单步和 trajectory 评估使用干净、预格式化的输入来测试 Agent。真实的客户交互更加混乱——它们跨越多个回合，包括歧义，并且需要 Agent 处理真实对话的流程，包括 HITL 验证步骤。

<div align="center">
    <img src="../../images/simulation-eval.png"  width="800">
</div>

**多轮模拟**通过用 LLM 驱动的模拟用户代替人类来解决这个问题，从头开始生成真实的对话并评估完整的交互。

**我们正在测试的内容：** 与 `supervisor_hitl_agent` 的完整对话，包括：
- Agent 在需要时询问客户的电子邮件
- 客户自然地提供他们的电子邮件（用他们自己的话）
- Agent 检索正确的信息并做出有帮助的响应
- 应该完全跳过验证的情况

我们将使用提供模拟基础设施的 [`openevals`](https://github.com/langchain-ai/openevals) 库。

### 3.1 构建应用程序包装器

`run_multiturn_simulation` 函数一次调用我们的应用程序一条消息。每个调用都会收到一条消息，并应返回一个响应。

关键挑战：LangGraph 的 HITL 使用 `interrupt()` 在运行中暂停图形。当 Agent 等待电子邮件时，下一条用户消息应该**恢复**中断的图表 - 而不是开始新的运行。

我们通过在每次调用之前检查 `agent.get_state(config).next` 来处理此问题：
- 如果图表有待处理的步骤（活动 interrupt），则使用 `Command(resume=...)` 恢复
- 否则，使用新消息正常调用

In [ ]:
def make_app(agent):
    """Wrap a LangGraph HITL agent for use with run_multiturn_simulation.

    The simulator calls app(message, thread_id=...) once per turn.
    We detect active interrupts via get_state().next and route accordingly.
    """

    def app(inputs: dict, *, thread_id: str, **kwargs) -> dict:
        config = {"configurable": {"thread_id": thread_id}}
        state = agent.get_state(config)

        if state.next:  # graph is paused at an interrupt (waiting for email)
            result = agent.invoke(Command(resume=inputs["content"]), config=config)
        else:  # fresh turn — invoke normally
            result = agent.invoke({"messages": [inputs]}, config=config)

        return {"role": "assistant", "content": result["messages"][-1].content}

    return app

### 3.2 创建Dataset

每个示例都为模拟用户定义了一个**客户角色**，并为 evaluator 定义了**成功标准**。角色塑造了模拟用户的行为方式——他们何时提供电子邮件、他们有多沮丧等等。

In [ ]:
simulation_examples = [
    {
        "simulated_user_prompt": (
            "You are a TechHub customer named Sarah Chen. Your email is sarah.chen@gmail.com. "
            "You want to find out exactly which items were in your most recent order and how much each item cost. "
            "Only provide your email address when the agent explicitly asks for it."
        ),
        "success_criteria": (
            "The agent asked for Sarah's email to verify her identity, successfully verified it, "
            "and provided a breakdown of the items and costs from her most recent order."
        ),
    },
    {
        "simulated_user_prompt": (
            "You are a frustrated TechHub customer. Your email is james.nguyen@yahoo.com. "
            "You placed an order about a week ago and it still hasn't shipped — you're annoyed and want to know what's going on. "
            "Start by asking about the order status without giving your email. "
            "When the agent asks for your email, provide it."
        ),
        "success_criteria": (
            "The agent verified James's identity, looked up the order status, "
            "and responded professionally and empathetically to his concern about the delayed shipment."
        ),
    },
    {
        "simulated_user_prompt": (
            "You are a TechHub customer who is considering buying a new laptop. "
            "You want to understand the return policy before purchasing. "
            "You have not made any purchases yet and prefer not to share personal information unless absolutely necessary."
        ),
        "success_criteria": (
            "The agent answered the return policy question clearly and directly without requesting "
            "identity verification, since no account information was needed."
        ),
    },
]

simulation_dataset_name = f"techhub-multiturn-simulation-{uuid.uuid4()}"

simulation_dataset = client.create_dataset(
    dataset_name=simulation_dataset_name,
    description="Multi-turn simulation: LLM-simulated customer conversations with HITL verification",
)

client.create_examples(
    dataset_id=simulation_dataset.id,
    inputs=[
        {"simulated_user_prompt": ex["simulated_user_prompt"]}
        for ex in simulation_examples
    ],
    outputs=[
        {"success_criteria": ex["success_criteria"]} for ex in simulation_examples
    ],
)

print(f"Dataset: {simulation_dataset.url}")

### 3.3 目标函数

目标函数根据 dataset 中的角色创建一个新的模拟用户，然后运行模拟。返回的 `trajectory` 是对话中消息的完整列表。

In [ ]:
from openevals.simulators import create_llm_simulated_user, run_multiturn_simulation

# Re-create the agent instance for simulation (needs a fresh checkpointer state)
simulation_agent = create_supervisor_hitl_agent(
    db_agent=create_sql_agent(),
    docs_agent=create_docs_agent(),
)


def run_simulation(inputs: dict) -> dict:
    """Run a multi-turn simulation using the customer persona from the dataset."""
    user = create_llm_simulated_user(
        system=inputs["simulated_user_prompt"],
        model=DEFAULT_MODEL,
    )
    result = run_multiturn_simulation(
        app=make_app(simulation_agent),
        user=user,
        max_turns=6,
    )
    return {"trajectory": result["trajectory"]}

### 3.4 评估者

多轮模拟输出本质上是可变的——每次运行都会产生不同的对话。 LLM 法官评估器非常适合这里，因为他们可以整体评估对话质量，而不是检查特定字符串。

我们将使用两个评估器：
- **`resolution`** — Agent 是否满足 dataset 中定义的成功标准？
- **`num_turns`** — 转了多少圈？ （更低=更高效）

In [ ]:
from openevals.llm import create_llm_as_judge

resolution_evaluator = create_llm_as_judge(
    model=DEFAULT_MODEL,
    prompt=(
        "You are evaluating a customer support conversation.\n\n"
        "Success criteria:\n{reference_outputs}\n\n"
        "Conversation transcript:\n{outputs}\n\n"
        "Did the agent successfully meet the success criteria? "
        "Answer True if yes, False if not."
    ),
    feedback_key="resolution",
)


def num_turns(outputs: dict, **kwargs) -> dict:
    """Count the number of conversation turns (user + assistant pairs)."""
    return {"key": "num_turns", "score": len(outputs["trajectory"]) // 2}

### 3.5 运行模拟

In [ ]:
simulation_results = client.evaluate(
    run_simulation,
    data=simulation_dataset_name,
    evaluators=[resolution_evaluator, num_turns],
    experiment_prefix="multiturn-simulation",
    description="Multi-turn simulation: LLM-simulated customer personas with HITL verification",
    max_concurrency=1,
)

### 3.6 分析结果

打开上面的 experiment 链接并查看模拟trace。每个 trace 都会显示完整的模拟对话 - 您可以阅读客户和 Agent 之间的来回对话。

**要寻找的关键事项：**
- Agent 是否在需要时要求进行电子邮件验证（示例 1 和 2），并在不需要时跳过它（示例 3）？
- Agent 在真实对话中处理电子邮件请求的自然程度如何？
- `num_turns` 是否保持合理（2-4 回合）？高轮数可能表明 Agent 正在询问不必要的后续问题。
- 对于沮丧的客户（示例 2）：Agent 是同情地还是机械地回应？

**为什么模拟会捕获 trajectory 评估未命中的内容：**  
Trajectory 评估测试干净、预格式化的输入。使用凌乱、真实的消息进行模拟测试 - “是的，这是 james.nguyen@yahoo.com”与干净提取的电子邮件不同，Agent 必须处理这两者。

---
## 总结

您现在已经为 TechHub Agent 构建了全面的 Evaluation 套件：

|部分|评估类型 |它测量什么 | Experiment 前缀 |
|---|---|---|---|
|模块 2，S1 |最终答案|响应的正确性 | `baseline-eval` |
|模块 2，S2 |最终答案| SQL Agent 的改进 | `with-sql-agent-eval` |
| **本节** |单步| Supervisor 路由决策| `supervisor-routing-single-step` |
| **本节** | Trajectory |Tool Calling序列+HITL路径| `full-agent-trajectory` |
| **本节** |模拟|真实的多轮对话 | `multiturn-simulation` |

**何时使用每个：**
- **最终答案**：在任何更改后验证正确性。捕捉回归。
- **单步**：调试特定决策点。有针对性的修复的快速反馈循环。
- **Trajectory**：在最终答案中发现效率低下、缺少步骤或错误路径之前。
- **模拟**：测试真实对话，包括边缘情况和多轮动态。在发布到生产环境之前运行。